# Tarea 2 – Parte 2: Evaluación y extracción estructurada
Configuraciones fijadas ANTES de ver métricas:
- OCR base: PaddleOCR PP-OCRv5 mobile (Parte 1) | OCR alternativo: EasyOCR (es, CPU) — elegido por instalarse solo con pip y ser otra arquitectura (CRAFT + CRNN).
- ASR base: faster-whisper `small` CPU int8 + VAD | ASR alternativo: MMS-1b-all (spa), modelo CTC multilingüe de Meta.
- Normalización (referencias y salidas por igual): seudonimización, quitar marcas [..], minúsculas, sin tildes ni puntuación.
- LLM: gemini-2.5-flash, temperatura 0, salida JSON.

In [1]:
import os, re, json, time, glob, unicodedata, datetime, getpass
import pandas as pd, plotly.express as px, jiwer
import av
_av_open = av.open
def _av_open_compat(*a, **k):
    k.pop("metadata_errors", None); return _av_open(*a, **k)
av.open = _av_open_compat

docs = pd.read_csv("datos/documentos.csv")
trans = pd.read_csv("datos/transcripciones.csv")

try:
    import spacy; nlp = spacy.load("es_core_news_md")
except Exception:
    nlp = None
def seudonimizar(t):
    t = "" if pd.isna(t) else str(t)
    for pat, et in [(r"\S+@\S+", "EMAIL"), (r"\b9\d{8}\b", "TEL"), (r"\b\d{8}\b", "DNI")]:
        t = re.sub(pat, f"[{et}]", t)
    if nlp:
        for e in reversed(nlp(t).ents):
            if e.label_ == "PER": t = t[:e.start_char] + "[PERSONA]" + t[e.end_char:]
    return t

def norm(t):
    t = re.sub(r"\[[^\]]*\]", " ", "" if pd.isna(t) else str(t))   # quita [HABLANTE_1], [PERSONA]...
    t = unicodedata.normalize("NFKD", t.lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def error(fn, ref, hyp):
    ref, hyp = norm(ref), norm(hyp)
    if not ref: return None
    return fn(ref, hyp) if hyp else 1.0

In [2]:
muestra_docs = pd.concat([docs[docs.tipo == t].sample(n, random_state=42)
                          for t, n in [("impresos", 3), ("fotos", 3), ("manuscritos", 4)]])
muestra_aud = trans.sample(10, random_state=42)
os.makedirs("data/referencias", exist_ok=True)      # carpeta ignorada por Git
REF_D, REF_A = "data/referencias/ref_docs.xlsx", "data/referencias/ref_audios.xlsx"
if not os.path.exists(REF_D):
    muestra_docs[["id", "archivo", "tipo"]].assign(texto_referencia="").to_excel(REF_D, index=False)
    muestra_aud[["id", "archivo_audio"]].assign(condicion="", texto_referencia="").to_excel(REF_A, index=False)
    print("Plantillas creadas. Llénalas a mano antes de seguir.")

Plantillas creadas. Llénalas a mano antes de seguir.


In [3]:
ref_d, ref_a = pd.read_excel(REF_D), pd.read_excel(REF_A)
ref_d["ref"] = ref_d.texto_referencia.apply(seudonimizar)
ref_a["ref"] = ref_a.texto_referencia.apply(seudonimizar)
docs["texto_referencia"] = docs.id.map(dict(zip(ref_d.id, ref_d.ref)))
trans["texto_referencia"] = trans.id.map(dict(zip(ref_a.id, ref_a.ref)))
docs.to_csv("datos/documentos.csv", index=False, encoding="utf-8-sig")
trans.to_csv("datos/transcripciones.csv", index=False, encoding="utf-8-sig")
print(ref_d.ref.str.len().describe(), ref_a.ref.str.len().describe())

count      10.000000
mean     3150.600000
std      1938.171601
min      1106.000000
25%      1849.000000
50%      2813.000000
75%      3157.250000
max      7424.000000
Name: ref, dtype: float64 count      10.0000
mean      703.0000
std       719.7268
min         0.0000
25%         0.0000
50%       724.5000
75%      1160.0000
max      2118.0000
Name: ref, dtype: float64


In [4]:
import easyocr
reader = easyocr.Reader(["es"], gpu=False)
filas = []
for r in ref_d.itertuples():
    fila = docs[docs.id == r.id].iloc[0]
    filas.append({"id": r.id, "modalidad": "documento", "tipo": r.tipo, "modelo": "PaddleOCR PP-OCRv5 mobile",
                  "hipotesis": fila.texto_ocr, "referencia": r.ref})
    t0 = time.time()
    txt = " ".join(reader.readtext(fila.archivo, detail=0, paragraph=True))
    filas.append({"id": r.id, "modalidad": "documento", "tipo": r.tipo, "modelo": "EasyOCR",
                  "hipotesis": seudonimizar(txt), "referencia": r.ref, "latencia_seg": round(time.time() - t0, 1)})
det_docs = pd.DataFrame(filas)
det_docs["error"] = [error(jiwer.cer, a, b) for a, b in zip(det_docs.referencia, det_docs.hipotesis)]
det_docs.groupby(["modelo", "tipo"]).error.mean().unstack().round(3)

Using CPU. Note: This module is much faster with a GPU.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

c:\TAREA 2\.venv\Lib\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(
c:\TAREA 2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\TAREA 2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)

tipo,fotos,impresos,manuscritos
modelo,,,
EasyOCR,0.117,0.891,0.584
PaddleOCR PP-OCRv5 mobile,0.583,0.675,0.451


In [6]:
VIEJO, NUEVO = "do2.mp3", "doc2.mp3"

# CSV de la Parte 1
trans["archivo_audio"] = trans.archivo_audio.replace(VIEJO, NUEVO)
trans.to_csv("datos/transcripciones.csv", index=False, encoding="utf-8-sig")
fuentes = pd.read_csv("datos/fuentes.csv")
fuentes["archivo"] = fuentes.archivo.replace(VIEJO, NUEVO)
fuentes.to_csv("datos/fuentes.csv", index=False, encoding="utf-8-sig")

# Plantilla de referencias
ref_a["archivo_audio"] = ref_a.archivo_audio.replace(VIEJO, NUEVO)
ref_a.drop(columns="ref").to_excel(REF_A, index=False)

# Archivos de transcripción de la Parte 1
for ext in (".srt", ".txt"):
    viejo = f"outputs/asr/do2{ext}"
    if os.path.exists(viejo):
        os.replace(viejo, f"outputs/asr/doc2{ext}")
print("Corregido:", (trans.archivo_audio == NUEVO).sum(), "fila(s)")

Corregido: 1 fila(s)


In [7]:
from faster_whisper import WhisperModel
from transformers import Wav2Vec2ForCTC, AutoProcessor
import librosa, torch
whisper = WhisperModel("small", device="cpu", compute_type="int8")
proc = AutoProcessor.from_pretrained("facebook/mms-1b-all", target_lang="spa")
mms = Wav2Vec2ForCTC.from_pretrained("facebook/mms-1b-all", target_lang="spa", ignore_mismatched_sizes=True)

def asr_whisper(p):
    segs, _ = whisper.transcribe(p, language="es", vad_filter=True)
    return " ".join(s.text.strip() for s in segs)
def asr_mms(p):
    y, _ = librosa.load(p, sr=16000)
    partes = []
    for i in range(0, len(y), 16000 * 30):           # trozos de 30 s
        x = proc(y[i:i + 16000 * 30], sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            partes.append(proc.decode(mms(**x).logits.argmax(-1)[0]))
    return " ".join(partes)

filas = []
for r in ref_a.itertuples():
    p = os.path.join("data/audio", r.archivo_audio)
    dur = trans.loc[trans.id == r.id, "duracion_seg"].iloc[0]
    for modelo, fn in [("faster-whisper small", asr_whisper), ("MMS-1b-all", asr_mms)]:
        t0 = time.time(); txt = fn(p); dt = time.time() - t0
        filas.append({"id": r.id, "modalidad": "audio", "tipo": r.condicion, "modelo": modelo,
                      "hipotesis": seudonimizar(txt), "referencia": r.ref,
                      "latencia_seg": round(dt, 1), "rtf": round(dt / dur, 2)})
det_aud = pd.DataFrame(filas)
det_aud["error"] = [error(jiwer.wer, a, b) for a, b in zip(det_aud.referencia, det_aud.hipotesis)]
det_aud.groupby(["modelo", "tipo"]).error.mean().unstack().round(3)

Loading weights: 100%|██████████| 1096/1096 [00:19<00:00, 57.20it/s]


tipo,"\nEl Servicio Militar Voluntario desde pequeña fue mi anhelo pertenecer aquí a la Fuerza Aérea, que es una bonita institución.\n\nA los jóvenes de hoy en día yo les diría pues que el Servicio Militar es una llave que te abre muchas puertas en sí, te abre nuevas oportunidades, te ayuda bastante en sí.\n\nEstoy estudiando en el Instituto ISAM.\n\nVoy por cuarto ciclo la carrera de Administración de Empresas.\n\nMe siento muy contenta.\n\nAgradezco al Ministerio de Defensa por el apoyo económico\n\ny ahora, gracias a ello, voy a solventar totalmente\n\nel pago de mi pensión de estudios.\n\nEs mi primer ciclo y mi carrera es Ingeniería de Ciberseguridad.\n\nLa propina va a ayudar tanto en mi estudio,\n\ntanto en mis materiales, en pasaje,\n\nen el lugar donde yo me voy a quedar, me apoyan.\n\nMayormente también, pues, cosas personales también, es cierto.\n\nLas labores son de lunes a viernes.\n\nTodos los días son apoyo-oficina.\n\nO cuando estamos de monitores,\n\nestamos aquí con los reclutas que son recién ingresantes en la esplanada,\n\ndando instrucción, enseñando todo lo que ya nosotros tenemos conocimiento.\n\nEstoy estudiando virtualmente en las noches.\n\ny de día normal, este, laboro, apoyo con mi labor y en la noche estudio.\n\nMis exámenes también lo tomo virtualmente.\n\nMe gusta apoyar y por eso elegí el servicio militar voluntario.\n\nDesde pequeña fue mi anhelo pertenecer aquí a la Fuerza Aérea, que es una bonita institución.\n\nEl servicio militar es una llave que te abre muchas puertas,\n\nte abre nuevas oportunidades, te ayuda bastante en sí.\n\nEconómicamente, como ahora ya está aumentando nuestra rutina, te ayuda a sustentar.","Asistir al colegio ha sido un desafío que ha enfrentado durante muchos años Luz Janet\nVillafuerte de 14 años, pero su profundo deseo de estudiar la impulsó a caminar 8\nhoras para llegar al colegio y otras 8 horas para regresar a casa.\nSiempre me ha parecido dificultoso, hasta hay veces desmotivaba las ganas de\nquerer venir a este centro institucional porque era demasiado lejos\ndel anexo de Azabamba hasta el anexo de San Antonio\nEstá un aproximado de ocho horas.\nVenía caminando, siempre dificultaba.\nVenir en las movilidades también es un gran costo.\nCon el amor y apoyo de sus padres, esta valiente jovencita emprendía cada 15 días\nel largo y extenuante viaje, desde su casa en el anexo de Azabamba\nhasta el Centro Rural de Formación en Alternancia San Antonio,\nen el distrito de Chipau, Lucanas, Ayacucho.\nGracias a la intervención de FONCO desde el Ministerio de Desarrollo e Inclusión Social,\nla Escuela de Luz Janer y sus 87 compañeros de diversas comunidades,\nse ha convertido en una moderna construcción que ofrece todas las comodidades necesarias\npara brindar una educación de calidad a estos jóvenes.\nMe siento motivada. Ahora la comodidad es mejor.\nIncluso muchos estudiantes de otras instituciones quieren cambiarse a esta institución por la nueva infraestructura.\nCon las mejoras en las condiciones de la residencia, Luz Jeanette siente que su sueño está más cerca de hacerse realidad.\nYo quiero ser abogada porque me gusta bastante todo lo que es derechos.\nsaber comunicarse bien, tener una buena expresión.\nTengo bastantes esperanzas que es una buena oportunidad de conseguir mis sueños\nporque también tengo pocos ingresos económicos.\nLuz yanet es un ejemplo inspirador de cómo la inversión en educación\npuede generar un impacto significativo en la vida de quienes más lo necesitan.\nY ahora, llena de motivación, esta jovencita avanza, paso a paso, hacia sus metas que le permitirán alcanzar un futuro con mejor calidad de vida.\nMinisterio de Desarrollo e Inclusión Social.\nPonle punche y ganamos todos Perú.\nGobierno del Perú.","Bienvenidos a Midis Noticias.\nEsta semana fortalecimos la preparación ante posibles emergencias\ny reforzamos las acciones para proteger a las familias más vulnerables\ny prevenir la anemia en distintas regiones del país.\nEstas son las princi

In [10]:
ref_a = pd.read_excel(REF_A)
ref_a["ref"] = ref_a.texto_referencia.apply(seudonimizar)
trans["texto_referencia"] = trans.id.map(dict(zip(ref_a.id, ref_a.ref)))
trans.to_csv("datos/transcripciones.csv", index=False, encoding="utf-8-sig")

det_aud["referencia"] = det_aud.id.map(dict(zip(ref_a.id, ref_a.ref)))
det_aud["tipo"] = det_aud.id.map(dict(zip(ref_a.id, ref_a.condicion)))
det_aud["error"] = [error(jiwer.wer, a, b) for a, b in zip(det_aud.referencia, det_aud.hipotesis)]
print("Filas sin referencia:", det_aud.error.isna().sum())
det_aud.groupby(["modelo", "tipo"]).error.mean().unstack().round(3)

Filas sin referencia: 0


tipo,Limpio
modelo,
MMS-1b-all,0.148
faster-whisper small,0.054


In [11]:
det = pd.concat([det_docs, det_aud], ignore_index=True)
det.to_csv("datos/metricas_detalle.csv", index=False, encoding="utf-8-sig")
tabla = (det.groupby(["modalidad", "modelo"])
            .agg(valor=("error", "mean"), latencia_seg=("latencia_seg", "mean"), rtf=("rtf", "mean"))
            .round(3).reset_index())
tabla["metrica"] = tabla.modalidad.map({"documento": "CER", "audio": "WER"})
display(tabla)
display(det.groupby(["modelo", "tipo"]).error.mean().unstack().round(3))   # casos difíciles
mejor = tabla.loc[tabla.groupby("modalidad").valor.idxmin()]
px.bar(tabla, x="modelo", y="valor", color="modalidad", barmode="group",
       title=f"Menor error: {' y '.join(mejor.modelo)}").show()

,modalidad,modelo,valor,latencia_seg,rtf,metrica
0,audio,MMS-1b-all,0.148,169.92,1.455,WER
1,audio,faster-whisper small,0.054,47.72,0.406,WER
2,documento,EasyOCR,0.536,20.28,NaN,CER
3,documento,PaddleOCR PP-OCRv5 mobile,0.558,NaN,NaN,CER


tipo,Limpio,fotos,impresos,manuscritos
modelo,,,,
EasyOCR,NaN,0.117,0.891,0.584
MMS-1b-all,0.148,NaN,NaN,NaN
PaddleOCR PP-OCRv5 mobile,NaN,0.583,0.675,0.451
faster-whisper small,0.054,NaN,NaN,NaN


In [2]:
det = pd.read_csv("datos/metricas_detalle.csv")
tabla = (det.groupby(["modalidad", "modelo"])
            .agg(valor=("error", "mean"), latencia_seg=("latencia_seg", "mean"), rtf=("rtf", "mean"))
            .round(3).reset_index())
tabla["metrica"] = tabla.modalidad.map({"documento": "CER", "audio": "WER"})
tabla

,modalidad,modelo,valor,latencia_seg,rtf,metrica
0,audio,MMS-1b-all,0.148,169.92,1.455,WER
1,audio,faster-whisper small,0.054,47.72,0.406,WER
2,documento,EasyOCR,0.536,20.28,NaN,CER
3,documento,PaddleOCR PP-OCRv5 mobile,0.558,NaN,NaN,CER


In [4]:
from google import genai
if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Clave Gemini: ")
cliente_lista = genai.Client()
modelos = [m.name for m in cliente_lista.models.list()
           if "generateContent" in (m.supported_actions or [])]
print("\n".join(modelos))

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/l

In [5]:
from google import genai
from google.genai import types
if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Clave Gemini: ")
client = genai.Client(http_options=types.HttpOptions(timeout=60_000))   # máx. 60 s por petición
MODELO_LLM = "gemini-3.5-flash-lite"
PROMPT = """Lee este texto (salida de OCR o de transcripción; puede tener errores) y responde SOLO un objeto JSON con estas claves:
{campos}
Reglas: usa null si el dato no aparece explícitamente en el texto; no inventes datos ni agregues claves; "resumen" en máximo 2 oraciones en español.
TEXTO:
{texto}"""
CAMPOS = {"documento": '"modalidad" (="documento"), "tema", "resumen", "emisor", "fecha", "monto"',
          "audio": '"modalidad" (="audio"), "tema", "resumen", "citas_clave" (lista de frases literales), "numero_hablantes"'}
items = [("documento", r.id, r.texto_ocr) for r in docs.itertuples()] + \
        [("audio", r.id, r.texto_asr) for r in trans.itertuples()]
textos = {i: t for _, i, t in items}

SALIDA = "datos/extraccion_llm.csv"
ok = {}
if os.path.exists(SALIDA):                     # continuar donde quedó
    prev = pd.read_csv(SALIDA)
    prev = prev[(prev.modelo_llm == MODELO_LLM) & ~prev.campos_json.str.contains('"resumen": null')]
    ok = {r["id"]: r for r in prev.to_dict("records")}
pendientes = [it for it in items if it[1] not in ok]
print(f"Ya listos: {len(ok)} | Pendientes: {len(pendientes)}")

for n, (mod, id_, texto) in enumerate(pendientes, 1):
    campos = {"modalidad": mod, "tema": None, "resumen": None}
    for intento in range(3):
        try:
            resp = client.models.generate_content(
                model=MODELO_LLM, contents=PROMPT.format(campos=CAMPOS[mod], texto=str(texto)[:6000]),
                config={"response_mime_type": "application/json", "temperature": 0})
            campos = json.loads(resp.text); break
        except Exception as e:
            print(f"  {id_} intento {intento + 1} falló: {str(e)[:70]}"); time.sleep(20 * (intento + 1))
    ok[id_] = {"id": id_, "modalidad": mod, "campos_json": json.dumps(campos, ensure_ascii=False),
               "modelo_llm": MODELO_LLM, "fecha": datetime.date.today().isoformat()}
    llm = pd.DataFrame([ok[i] for _, i, _ in items if i in ok])
    llm.to_csv(SALIDA, index=False, encoding="utf-8-sig")
    print(f"[{n}/{len(pendientes)}] {id_} {'OK' if campos.get('resumen') else 'FALLÓ'}")
    time.sleep(7)

llm = pd.read_csv(SALIDA)
print("Total:", len(llm), "| Siguen fallando:", llm.campos_json.str.contains('"resumen": null').sum())

Ya listos: 0 | Pendientes: 45
[1/45] D001 OK
[2/45] D002 OK
[3/45] D003 OK
[4/45] D004 OK
[5/45] D005 OK
[6/45] D006 OK
[7/45] D007 OK
[8/45] D008 OK
[9/45] D009 OK
[10/45] D010 OK
[11/45] D011 OK
[12/45] D012 OK
[13/45] D013 OK
[14/45] D014 OK
[15/45] D015 OK
[16/45] D016 OK
[17/45] D017 OK
[18/45] D018 OK
[19/45] D019 OK
[20/45] D020 OK
[21/45] D021 OK
[22/45] D022 OK
[23/45] D023 OK
[24/45] D024 OK
[25/45] D025 OK
[26/45] A001 OK
[27/45] A002 OK
[28/45] A003 OK
[29/45] A004 OK
[30/45] A005 OK
[31/45] A006 OK
[32/45] A007 OK
[33/45] A008 OK
[34/45] A009 OK
[35/45] A010 OK
[36/45] A011 OK
[37/45] A012 OK
[38/45] A013 OK
[39/45] A014 OK
[40/45] A015 OK
[41/45] A016 OK
[42/45] A017 OK
[43/45] A018 OK
[44/45] A019 OK
[45/45] A020 OK
Total: 45 | Siguen fallando: 0


### Criterios de auditoría
Una extracción es **completamente correcta** si:
1. `tema` y `resumen` reflejan fielmente el texto fuente.
2. Todo campo con valor (emisor, fecha, monto, citas_clave, numero_hablantes) aparece explícitamente en el texto.
3. Los campos ausentes en el texto están como `null` (sin invenciones).
Un solo error en cualquier campo la vuelve incorrecta.

In [7]:
AUD = "data/referencias/auditoria.xlsx"
if not os.path.exists(AUD):
    textos = dict(zip(docs.id, docs.texto_ocr)) | dict(zip(trans.id, trans.texto_asr))
    m = pd.concat([llm[llm.modalidad == "documento"].sample(10, random_state=1),
                   llm[llm.modalidad == "audio"].sample(10, random_state=1)])
    m.assign(texto_fuente=m.id.map(textos), correcto="", comentario="").to_excel(AUD, index=False)
    print("Llena 'correcto' (1 o 0) en el Excel y vuelve a ejecutar esta celda.")
else:
    aud = pd.read_excel(AUD)
    acuerdo = int(aud.correcto.sum())
    print(f"Acuerdo LLM-Humano: {acuerdo}/20")
    display(aud[aud.correcto == 0][["id", "campos_json", "comentario"]].head(5))   # ejemplos de error

Acuerdo LLM-Humano: 16/20


,id,campos_json,comentario
2,D018,"{""modalidad"": ""documento"", ""tema"": ""Evolución ...",no muy fiel. Errores de spelling
4,D022,"{""modalidad"": ""documento"", ""tema"": ""El concept...",no muy fiel. Errores de spelling
6,D019,"{""modalidad"": ""documento"", ""tema"": ""Relaciones...",no muy fiel. Errores de spelling
14,A003,"{""modalidad"": ""audio"", ""tema"": ""desarrollo e i...",no pudo transcribir


In [10]:
# Recupera todo desde disco (no depende de variables en memoria)
det = pd.read_csv("datos/metricas_detalle.csv")
tabla = (det.groupby(["modalidad", "modelo"])
            .agg(valor=("error", "mean"), latencia_seg=("latencia_seg", "mean"), rtf=("rtf", "mean"))
            .round(3).reset_index())
tabla["metrica"] = tabla.modalidad.map({"documento": "CER", "audio": "WER"})

llm = pd.read_csv("datos/extraccion_llm.csv")
MODELO_LLM = llm.modelo_llm.iloc[0]
acuerdo = int(pd.read_excel("data/referencias/auditoria.xlsx").correcto.sum())
print(f"LLM: {MODELO_LLM} | Acuerdo LLM-Humano: {acuerdo}/20")

met = tabla[["modalidad", "modelo", "metrica", "valor", "latencia_seg", "rtf"]]
met = pd.concat([met, pd.DataFrame([{"modalidad": "llm", "modelo": MODELO_LLM, "metrica": "acuerdo", "valor": acuerdo / 20}])])
met.to_csv("datos/metricas.csv", index=False, encoding="utf-8-sig")

os.makedirs("datos/ejemplos/asr", exist_ok=True)
for f in glob.glob("outputs/asr/*.srt"):
    lineas = open(f, encoding="utf-8").read().splitlines()
    lineas = [seudonimizar(l) if l.startswith("[") else l for l in lineas]
    open(os.path.join("datos/ejemplos/asr", os.path.basename(f)), "w", encoding="utf-8").write("\n".join(lineas))
print("SRT seudonimizados:", len(glob.glob("datos/ejemplos/asr/*.srt")))
met

LLM: gemini-3.5-flash-lite | Acuerdo LLM-Humano: 16/20
SRT seudonimizados: 20


,modalidad,modelo,metrica,valor,latencia_seg,rtf
0,audio,MMS-1b-all,WER,0.148,169.92,1.455
1,audio,faster-whisper small,WER,0.054,47.72,0.406
2,documento,EasyOCR,CER,0.536,20.28,NaN
3,documento,PaddleOCR PP-OCRv5 mobile,CER,0.558,NaN,NaN
0,llm,gemini-3.5-flash-lite,acuerdo,0.800,NaN,NaN




## Reflexiones sobre el desempeño de los modelos

### Modelos de audio

faster-whisper small obtuvo el mejor desempeño entre los modelos de audio evaluados. Su WER de 0.054 indica una tasa de error de palabras de aproximadamente 5.4 %, inferior al 14.8 % registrado por MMS-1b-all. Además, presentó una latencia de 47.72 segundos, frente a los 169.92 segundos de MMS-1b-all, lo que representa una reducción aproximada del 72 % en el tiempo de procesamiento.

ofrece un mejor equilibrio entre precisión y velocidad en este conjunto de pruebas. Su RTF de 0.406 indica que procesa el audio en menos tiempo que su duración, mientras que MMS-1b-all, con un RTF de 1.455, tarda más que el tiempo real del audio.

Estos resultados pueden variar según la duración y calidad de la grabación, el ruido ambiental, los acentos y la velocidad del habla. 

### Modelos de reconocimiento de documentos

EasyOCR obtuvo un CER de 0.536, ligeramente inferior al 0.558 de PaddleOCR PP-OCRv5 mobile. La diferencia es de 2.2 puntos porcentuales, por lo que EasyOCR presentó una pequeña ventaja en precisión según esta métrica.

Sin embargo, ambos valores de CER son elevados y sugieren errores importantes en el reconocimiento de caracteres. Entre las posibles causas se encuentran la baja resolución de las imágenes, el texto pequeño, la inclinación de las páginas, las sombras, las tipografías poco claras y las estructuras complejas, como tablas y columnas.

EasyOCR registró una latencia de 20.28 segundos. Como no se dispone de la latencia de PaddleOCR, no es posible determinar cuál de los dos modelos es más rápido. 

### Modelo de lenguaje

Gemini 3.5 Flash Lite obtuvo un acuerdo de 0.800, equivalente al 80 % si la métrica se expresa como proporción. Este resultado sugiere un desempeño razonable para la tarea evaluada, aunque todavía existe margen de mejora.

La interpretación depende de cómo se definió el acuerdo, por ejemplo, si se compararon respuestas completas, decisiones de clasificación o coincidencias con una respuesta de referencia. Por ello, es recomendable analizar los casos de desacuerdo para identificar errores recurrentes.

Como no se dispone de datos de latencia ni de costo computacional, no es posible determinar si este modelo representa la alternativa más eficiente. Su selección debe considerar tanto la calidad de las respuestas como los recursos necesarios para ejecutarlo.

## 5. Conclusión

Los resultados muestran que no existe un modelo universalmente superior para todas las modalidades. Faster-whisper small destacó en la transcripción de audio por su precisión y velocidad, mientras que EasyOCR presentó una ligera ventaja en el reconocimiento de documentos. Gemini 3.5 Flash Lite alcanzó un 80 % de acuerdo, aunque se necesita información adicional para valorar su eficiencia computacional.

